In [1]:
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, classification_report, \
confusion_matrix, precision_score, recall_score
from lightgbm import LGBMClassifier 
from sklearn.preprocessing import RobustScaler, MinMaxScaler
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
playlist_df = pd.read_csv("../data/raw/capstonehorror.csv",",")
playlist_df.head()

/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3553: FutureWarning: In a future version of pandas all arguments of read_csv except for the argument 'filepath_or_buffer' will be keyword-only
  exec(code_obj, self.user_global_ns, self.user_ns)


,Spotify ID,Artist IDs,Track Name,Album Name,Artist Name(s),Release Date,Duration (ms),Popularity,Added By,Added At,...,Key,Loudness,Mode,Speechiness,Acousticness,Instrumentalness,Liveness,Valence,Tempo,Time Signature
0,2LlQb7Uoj1kKyGhlkBf9aC,3fMbdgg4jU18AjLCKBhRSm,Thriller,Thriller,Michael Jackson,1982-11-30,357800,73,spotify:user:willdudek,2023-12-06T03:43:49Z,...,11,-9.814,1,0.0693,0.0318,0.000225,0.890,0.7830,118.392,4
1,0pBEs0bmZZifddxLCIIWO6,"3Ia5S5H9yoQpc9Hqy0cqAU,7zvtLcfWDs8WZGvvy99ciF,...",Main Theme - Desolation,The Thing - Music From The Motion Picture (Enn...,"Alan Howarth,Larry Hopkins,Ennio Morricone",2012-02-21,271080,0,spotify:user:willdudek,2023-12-06T03:44:50Z,...,5,-18.709,0,0.0286,0.6240,0.879000,0.107,0.0502,98.980,4
2,2FjW6XLCZCQORjl2RQpIMP,7t8q7ikEtcPNtoaKAm9Vu6,The Omen - Ave Santani,The Omen - Deluxe Edition,Jerry Goldsmith,1976,155760,0,spotify:user:willdudek,2023-12-06T03:45:01Z,...,5,-15.569,0,0.0355,0.5810,0.008080,0.145,0.1090,144.179,1
3,6Yu1AkzMsuyznelPhhBi71,1LvnYiL8t79jMHt74r8El9,Eaten Alive! - Original Soundtrack from Eaten ...,Eaten Alive! (Original Soundtrack from Eaten A...,Roberto Donati,2012-11-20,129097,21,spotify:user:willdudek,2023-12-06T03:45:30Z,...,10,-6.586,0,0.0655,0.0410,0.601000,0.214,0.7800,128.304,4
4,0znLXdMRKNazUtGb7MmfTh,"0YC192cP3KPCRWx8zr8MfZ,1rV5NNLfWbeGqkDHqpkL5p,...",Dear Clarice,Hannibal - Original Motion Picture Soundtrack,"Hans Zimmer,Anthony Hopkins,Libera,The Lyndhur...",2001-01-01,361250,0,spotify:user:willdudek,2023-12-06T03:45:43Z,...,9,-23.762,0,0.0458,0.5380,0.519000,0.085,0.0384,168.764,3


In [3]:
playlist_df.columns

Index(['Spotify ID', 'Artist IDs', 'Track Name', 'Album Name',
       'Artist Name(s)', 'Release Date', 'Duration (ms)', 'Popularity',
       'Added By', 'Added At', 'Genres', 'Danceability', 'Energy', 'Key',
       'Loudness', 'Mode', 'Speechiness', 'Acousticness', 'Instrumentalness',
       'Liveness', 'Valence', 'Tempo', 'Time Signature'],
      dtype='object')

In [3]:
# Update columns names (lowercase, underscore, no special chars)
playlist_df.columns = playlist_df.columns.str.replace(' ','_').str.lower()
playlist_df.columns = playlist_df.columns.str.replace('(','').str.replace(')','')

/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:3: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  This is separate from the ipykernel package so we can avoid doing imports until


## Data Prep / Feature Engineering

In [4]:
# X_df = df[['duration_ms', 'danceability', 'energy', 'loudness', 'mode',
#        'speechiness', 'acousticness', 'instrumentalness', 'liveness',
#        'valence', 'tempo', 'time_signature']]
spotify_features_list = ['loudness','tempo','danceability','instrumentalness','acousticness','energy','valence']
X_df = playlist_df.loc[:,spotify_features_list]
X_df

,loudness,tempo,danceability,instrumentalness,acousticness,energy,valence
0,-9.814,118.392,0.771,0.000225,0.0318,0.687,0.7830
1,-18.709,98.980,0.559,0.879000,0.6240,0.120,0.0502
2,-15.569,144.179,0.230,0.008080,0.5810,0.133,0.1090
3,-6.586,128.304,0.613,0.601000,0.0410,0.857,0.7800
4,-23.762,168.764,0.182,0.519000,0.5380,0.123,0.0384
5,-12.725,106.283,0.752,0.000000,0.5000,0.445,0.9640
6,-5.749,73.122,0.576,0.000005,0.0296,0.719,0.7300
7,-7.911,89.846,0.595,0.000020,0.0129,0.804,0.5860
8,-14.745,129.012,0.723,0.094600,0.1110,0.490,0.7950


In [5]:
# Transform tempo column
X_df['tempo'] = X_df['tempo']/240

In [6]:
# Get features for each dataset
action_features = ['loudness', 'danceability', 'instrumentalness', 'acousticness', 'energy', 'valence']
comedy_features = ['loudness', 'acousticness', 'valence']
drama_features = ['loudness', 'danceability', 'instrumentalness', 'acousticness']
scifi_features = ['loudness', 'danceability', 'instrumentalness', 'acousticness','energy']
horror_features = ['instrumentalness', 'valence']
romance_features = ['tempo', 'danceability', 'instrumentalness', 'acousticness', 'valence']

# Slice datasets for different models
X_comedy = X_df.loc[:,comedy_features]
X_action = X_df.loc[:,action_features]
X_horror = X_df.loc[:,horror_features]
X_drama = X_df.loc[:,drama_features]
X_romance = X_df.loc[:,romance_features]
X_scifi = X_df.loc[:,scifi_features]

In [7]:
# Load scalers
action_genre_rb_scaler = pickle.load(open("action_genre_rb_scaler.pkl", "rb"))
comedy_genre_rb_scaler = pickle.load(open("comedy_genre_rb_scaler.pkl", "rb"))
drama_genre_rb_scaler = pickle.load(open("drama_genre_rb_scaler.pkl", "rb"))
horror_genre_rb_scaler = pickle.load(open("horror_genre_rb_scaler.pkl", "rb"))
romance_genre_rb_scaler = pickle.load(open("romance_genre_rb_scaler.pkl", "rb"))
scifi_genre_rb_scaler = pickle.load(open("scifi_genre_rb_scaler.pkl", "rb"))

In [8]:
# Get transformed datasets
X_action_transformed = action_genre_rb_scaler.transform(X_action)
X_comedy_transformed = comedy_genre_rb_scaler.transform(X_comedy)
X_drama_transformed = drama_genre_rb_scaler.transform(X_drama)
X_horror_transformed = horror_genre_rb_scaler.transform(X_horror)
X_romance_transformed = romance_genre_rb_scaler.transform(X_romance)
X_scifi_transformed = scifi_genre_rb_scaler.transform(X_scifi)

## Run genre classification models on songs and append back to dataframe

In [9]:
# Load action genre model 
loaded_model_action = pickle.load(open('music_to_action_final.pickle', "rb"))

# Load comedy genre model 
loaded_model_comedy = pickle.load(open('music_to_comedy_final.pickle', "rb"))

# Load sci-fi genre model 
loaded_model_scifi = pickle.load(open('music_to_scifi_final.pickle', "rb"))

# Load romance genre model 
loaded_model_romance = pickle.load(open('music_to_romance_final.pickle', "rb"))

# Load horror genre model 
loaded_model_horror = pickle.load(open('music_to_horror_final.pickle', "rb"))

# Load drama genre model 
loaded_model_drama = pickle.load(open('music_to_drama_final.pickle', "rb"))


In [10]:
# Predict genre likelihoods and attach back to dataframe
playlist_df['Action'] = loaded_model_action.predict_proba(X_action_transformed)[:,1]
playlist_df['Comedy'] = loaded_model_comedy.predict_proba(X_comedy_transformed)[:,1]
playlist_df['Sci-Fi'] = loaded_model_scifi.predict_proba(X_scifi_transformed)[:,1]
playlist_df['Romance'] = loaded_model_romance.predict_proba(X_romance_transformed)[:,1]
playlist_df['Horror'] = loaded_model_horror.predict_proba(X_horror_transformed)[:,1]
playlist_df['Drama'] = loaded_model_drama.predict_proba(X_drama_transformed)[:,1]

In [11]:
playlist_df

,spotify_id,artist_ids,track_name,album_name,artist_names,release_date,duration_ms,popularity,added_by,added_at,...,liveness,valence,tempo,time_signature,Action,Comedy,Sci-Fi,Romance,Horror,Drama
0,2LlQb7Uoj1kKyGhlkBf9aC,3fMbdgg4jU18AjLCKBhRSm,Thriller,Thriller,Michael Jackson,1982-11-30,357800,73,spotify:user:willdudek,2023-12-06T03:43:49Z,...,0.8900,0.7830,118.392,4,0.010447,0.994446,0.008062,0.000036,0.003790,0.928571
1,0pBEs0bmZZifddxLCIIWO6,"3Ia5S5H9yoQpc9Hqy0cqAU,7zvtLcfWDs8WZGvvy99ciF,...",Main Theme - Desolation,The Thing - Music From The Motion Picture (Enn...,"Alan Howarth,Larry Hopkins,Ennio Morricone",2012-02-21,271080,0,spotify:user:willdudek,2023-12-06T03:44:50Z,...,0.1070,0.0502,98.980,4,0.155703,0.083353,0.059944,0.000840,0.341785,0.706442
2,2FjW6XLCZCQORjl2RQpIMP,7t8q7ikEtcPNtoaKAm9Vu6,The Omen - Ave Santani,The Omen - Deluxe Edition,Jerry Goldsmith,1976,155760,0,spotify:user:willdudek,2023-12-06T03:45:01Z,...,0.1450,0.1090,144.179,1,0.090564,0.009600,0.069216,0.850129,0.072714,0.536898
3,6Yu1AkzMsuyznelPhhBi71,1LvnYiL8t79jMHt74r8El9,Eaten Alive! - Original Soundtrack from Eaten ...,Eaten Alive! (Original Soundtrack from Eaten A...,Roberto Donati,2012-11-20,129097,21,spotify:user:willdudek,2023-12-06T03:45:30Z,...,0.2140,0.7800,128.304,4,0.246176,0.998524,0.158083,0.000050,0.065195,0.137267
4,0znLXdMRKNazUtGb7MmfTh,"0YC192cP3KPCRWx8zr8MfZ,1rV5NNLfWbeGqkDHqpkL5p,...",Dear Clarice,Hannibal - Original Motion Picture Soundtrack,"Hans Zimmer,Anthony Hopkins,Libera,The Lyndhur...",2001-01-01,361250,0,spotify:user:willdudek,2023-12-06T03:45:43Z,...,0.0850,0.0384,168.764,3,0.733649,0.036437,0.355173,0.000743,0.720767,0.428656
5,3MtP9T0WF4xbIo8vqHYeXM,5zaXYwewAXedKNCff45U5l,Spooky,Dusty... Definitely,Dusty Springfield,1968-11-22,164173,73,spotify:user:willdudek,2023-12-06T03:46:03Z,...,0.0912,0.9640,106.283,4,0.009045,0.943422,0.010436,0.057180,0.000416,0.721146
6,5BBK1WoaiD3b3sJ2UysLuM,03uMw43UVu9MsQCcHVSGjX,Witches,Witches,Alice Phoebe Lou,2020-03-13,195078,66,spotify:user:willdudek,2023-12-06T03:46:07Z,...,0.1690,0.7300,73.122,4,0.090518,0.988570,0.067643,0.000118,0.002556,0.368218
7,6PNvv1dmDbOWrAYwEcuKBX,3g2kUQ6tHLLbmkV7T4GPtL,Criminal,Tidal,Fiona Apple,1996-07-23,341533,66,spotify:user:willdudek,2023-12-06T03:46:10Z,...,0.2990,0.5860,89.846,4,0.058690,0.803777,0.058117,0.000180,0.002766,0.039295
8,05oETzWbd4SI33qK2gbJfR,08GQAI4eElDnROBrJRGE0X,Rhiannon,Fleetwood Mac,Fleetwood Mac,1975-07-11,252773,78,spotify:user:willdudek,2023-12-06T03:46:19Z,...,0.0923,0.7950,129.012,4,0.015747,0.991483,0.087778,0.000063,0.049719,0.281948


In [12]:
df_condensed = playlist_df[["track_name","album_name","artist_names",
                            "Action","Comedy","Romance","Horror","Drama","Sci-Fi"]]
df_condensed

,track_name,album_name,artist_names,Action,Comedy,Romance,Horror,Drama,Sci-Fi
0,Thriller,Thriller,Michael Jackson,0.010447,0.994446,0.000036,0.003790,0.928571,0.008062
1,Main Theme - Desolation,The Thing - Music From The Motion Picture (Enn...,"Alan Howarth,Larry Hopkins,Ennio Morricone",0.155703,0.083353,0.000840,0.341785,0.706442,0.059944
2,The Omen - Ave Santani,The Omen - Deluxe Edition,Jerry Goldsmith,0.090564,0.009600,0.850129,0.072714,0.536898,0.069216
3,Eaten Alive! - Original Soundtrack from Eaten ...,Eaten Alive! (Original Soundtrack from Eaten A...,Roberto Donati,0.246176,0.998524,0.000050,0.065195,0.137267,0.158083
4,Dear Clarice,Hannibal - Original Motion Picture Soundtrack,"Hans Zimmer,Anthony Hopkins,Libera,The Lyndhur...",0.733649,0.036437,0.000743,0.720767,0.428656,0.355173
5,Spooky,Dusty... Definitely,Dusty Springfield,0.009045,0.943422,0.057180,0.000416,0.721146,0.010436
6,Witches,Witches,Alice Phoebe Lou,0.090518,0.988570,0.000118,0.002556,0.368218,0.067643
7,Criminal,Tidal,Fiona Apple,0.058690,0.803777,0.000180,0.002766,0.039295,0.058117
8,Rhiannon,Fleetwood Mac,Fleetwood Mac,0.015747,0.991483,0.000063,0.049719,0.281948,0.087778


In [13]:
# Get mean values
genre_stats = df_condensed.describe().loc['mean',:]
genre_stats

Action     0.156727
Comedy     0.649957
Romance    0.101038
Horror     0.139968
Drama      0.460938
Sci-Fi     0.097161
Name: mean, dtype: float64

In [15]:
# Get list of median values
median_vals = [df_condensed[val].median() for val in genre_stats.index]
median_vals

[0.09051813407464218,
 0.943421777485757,
 0.00017992668268483595,
 0.049719373434585504,
 0.4286563668904198,
 0.06764261310202334]

In [17]:
M

,mean,median,final_mean
Action,0.156727,0.090518,0.124
Comedy,0.649957,0.943422,0.797
Romance,0.101038,0.000180,0.051
Horror,0.139968,0.049719,0.095
Drama,0.460938,0.428656,0.445
Sci-Fi,0.097161,0.067643,0.082


In [ ]:
stats_df.describe()

In [17]:
cut_off_val = 0.1

# Extract genres that are below cutoff value (these kinds of movies will be removed)
null_genres = list(stats_df[stats_df.final_mean<cut_off_val].index)
null_genres

['Action', 'Romance', 'Horror']

## Read in movie data

In [18]:
movie_df = pd.read_csv('../data/cleaned/wiki_movies_mood_label_by_gpt-3.5-turbo_final.csv',',').loc[:,'year':]
movie_df

/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3553: FutureWarning: In a future version of pandas all arguments of read_csv except for the argument 'filepath_or_buffer' will be keyword-only
  exec(code_obj, self.user_global_ns, self.user_ns)


,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,sad,energetic,calm,title_stripped,year_from_wiki,imdb_id,omdb_poster,rated,omdb_runtime,Weighted_average
0,1973,The_Exorcist,The Exorcist,"In northern Iraq, priest Lankester Merrin tak...",NaN,NaN,Directed byWilliam Friedkin,Screenplay byWilliam Peter Blatty,Based onThe Exorcistby William Peter Blatty,Starring\nEllen Burstyn\nMax von Sydow\nLee J....,...,0.6,0.3,0.1,NaN,NaN,tt0070047,https://m.media-amazon.com/images/M/MV5BYWFlZG...,R,122,7.788206
1,1973,The_Sting,The Sting,"In 1936 Great Depression America, Johnny Hooke...",NaN,NaN,Directed byGeorge Roy Hill,NaN,NaN,Starring\nPaul Newman\nRobert Redford\nRobert ...,...,0.4,0.9,0.6,NaN,NaN,tt0070735,https://m.media-amazon.com/images/M/MV5BNGU3Nj...,PG,129,7.795403
2,1974,The_Godfather_Part_II,The Godfather Part II,The film interweaves between events some time ...,NaN,NaN,Directed byFrancis Ford Coppola,Screenplay by\nFrancis Ford Coppola\nMario Puzo\n,Based onThe Godfatherby Mario Puzo,Starring\nAl Pacino\nRobert Duvall\nDiane Keat...,...,0.7,0.8,0.3,NaN,NaN,tt0071562,https://m.media-amazon.com/images/M/MV5BMWMwMG...,R,202,8.827784
3,1974,Chinatown_(1974_film),Chinatown (1974 film),"In 1937, a woman identifying herself as Evelyn...",NaN,NaN,Directed byRoman Polanski,NaN,NaN,Starring\nJack Nicholson\nFaye Dunaway\nJohn H...,...,0.7,0.4,0.9,NaN,NaN,tt0071315,https://m.media-amazon.com/images/M/MV5BMjJkMD...,R,130,7.794692
4,1975,Jaws_(film),Jaws (film),"In the New England beach town of Amity Island,...",NaN,NaN,Directed bySteven Spielberg,Screenplay by\nPeter Benchley\nCarl Gottlieb\n,Based onJawsby Peter Benchley,Starring\nRoy Scheider\nRobert Shaw\nRichard D...,...,0.6,0.8,0.4,NaN,NaN,tt0073195,https://m.media-amazon.com/images/M/MV5BMmVmOD...,PG,124,7.873849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6466,2023,Haunting_of_the_Queen_Mary,Haunting of the Queen Mary,NaN,NaN,NaN,Directed byGary Shore,NaN,NaN,Starring\nAlice Eve\nJoel Fry\n,...,0.6,0.4,0.8,NaN,NaN,tt3463938,https://m.media-amazon.com/images/M/MV5BMDIwZW...,Not Rated,114,6.135713
6467,2023,The_Royal_Hotel_(film),The Royal Hotel (film),"Two young backpackers, Hanna and Liv, are trav...",NaN,NaN,Directed byKitty Green,NaN,Based onHotel Coolgardieby Pete Gleeson,Starring\nJulia Garner\nJessica Henwick\nToby ...,...,0.8,0.7,0.4,NaN,NaN,tt18363072,https://m.media-amazon.com/images/M/MV5BZDIzZG...,R,91,6.272443
6468,2023,Lee_(2023_film),Lee (2023 film),NaN,Lee Miller goes from a career as a model to en...,NaN,Directed byEllen Kuras,Screenplay by\nLiz Hannah\nJohn Collee\nMarion...,Based onThe Lives of Lee Millerby Antony Penrose,Starring\nKate Winslet\nAlexander Skarsgård\nA...,...,0.3,0.7,0.6,NaN,NaN,tt5112584,https://m.media-amazon.com/images/M/MV5BMjdjMj...,NaN,116,6.266684
6469,2023,The_Great_Escaper,The Great Escaper,NaN,NaN,NaN,Directed byOliver Parker,NaN,NaN,Starring\nMichael Caine\nGlenda Jackson\n,...,0.3,0.8,0.2,NaN,NaN,tt14124080,https://m.media-amazon.com/images/M/MV5BM2YxY2...,NaN,96,6.285265


In [19]:
movie_df.columns

Index(['year', 'href', 'title', 'plot', 'premise', 'synopsis', 'directed_by',
       'screenplay_by', 'based_on', 'starring', 'box_office', 'written_by',
       'budget', 'omdb_response', 'omdb_title', 'genres', 'omdb_plot',
       'release_date', 'omdb_director', 'omdb_writer', 'omdb_actors',
       'imdb_score', 'imdb_votes', 'rotten_tomatoes_score', 'metacritic_score',
       'plot_parenthetical_fuzzy_scores', 'new_plot', 'happy', 'sad',
       'energetic', 'calm', 'title_stripped', 'year_from_wiki', 'imdb_id',
       'omdb_poster', 'rated', 'omdb_runtime', 'Weighted_average'],
      dtype='object')

In [21]:
# Filter out movies of null genre
movie_df = movie_df[~movie_df['genres'].str.contains('|'.join(null_genres), case=False, na=False)]

In [22]:
# Verify that movies of null genres are gone
movie_df['genres_split'] = movie_df['genres'].str.split(', ')
all_genres = [genre for genres_list in movie_df['genres_split'].dropna() for genre in genres_list]
exploded_df = movie_df.explode('genres_split')
genre_counts = exploded_df['genres_split'].value_counts()
print(genre_counts)


Drama          2176
Comedy         1472
Crime           566
Adventure       451
Biography       451
Thriller        425
Family          331
Documentary     306
Mystery         299
Animation       272
Music           201
History         190
Fantasy         185
Sport           147
Sci-Fi           96
War              71
Short            65
Musical          52
Western          19
News              5
Reality-TV        3
Talk-Show         1
Name: genres_split, dtype: int64


/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
